## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import json
import shutil
from pathlib import Path

from entities import Portfolio
from config import PortfolioConfig
from utils import setup_logging, setup_random_seed
from utils.paths import EXPERIMENTS_DIR

logger = setup_logging()

In [ ]:
# ── Define which simulation to evaluate ──────────────────────
SIM_NAME = "trial06a50_w20seqd20h1002_n1000rand78"

sim_dir  = Path(EXPERIMENTS_DIR) / "simulations" / SIM_NAME
# load sim_meta
with open(sim_dir / "sim_meta.json") as f:
    sim_meta = json.load(f)

symbols = sim_meta["symbols"]
A       = sim_meta["n_assets"]

print(json.dumps(sim_meta, indent=2))

## 2. Load Simulation Results

In [ ]:
data = np.load(sim_dir / "sim_results.npz", allow_pickle=True)

# aggregated (cat) arrays — already prepared in inference notebook
actual_cat = data["actual_cat"]   # [A, T_total]
gen_cat    = data["gen_cat"]      # [n_paths, A, T_total]
gbm_paths  = data["gbm_paths"]   # [n_paths, A, T_total]
dates      = data["dates"]        # [N_windows, T]
close_prices = data["close_prices"]

# flat date index [T_total]
flat_dates = pd.to_datetime(dates.reshape(-1))
start_date = flat_dates[0].date()
end_date   = flat_dates[-1].date()

print(f"actual_cat : {actual_cat.shape}")
print(f"gen_cat    : {gen_cat.shape}")
print(f"gbm_paths  : {gbm_paths.shape}")
print(f"Date range : {start_date} → {end_date}")

## 3. Prep shapes for Portfolio

Portfolio expects:
- `gt_returns`  : `[T_total, A]`
- `mc_returns`  : `[n_paths, T_total, A]`

In [ ]:
actual_cat.shape

In [ ]:
W, A, T = close_prices.shape

close_prices_cat = close_prices.transpose(1, 0, 2).reshape(A, T * W)
close_prices_cat.shape

In [ ]:
# gt_returns : [T_total, A]
close_prices.T
gt_returns = actual_cat.T                         # [A, T] → [T, A]
gt_prices = close_prices_cat.T

# mc_returns : [n_paths, T_total, A]
mc_genai = gen_cat.transpose(0, 2, 1)             # [n_paths, A, T] → [n_paths, T, A]
mc_gbm   = gbm_paths.transpose(0, 2, 1)           # [n_paths, A, T] → [n_paths, T, A]

print(f"gt_returns : {gt_returns.shape}")
print(f"mc_genai   : {mc_genai.shape}")
print(f"mc_gbm     : {mc_gbm.shape}")

## 4. Portfolio Config

In [ ]:
cfg = PortfolioConfig()

cfg.annual_risk_free_rate = 0.02
cfg.rebalance_frequency = 10
cfg.optimize_window = 40
cfg.transaction_cost_rate = 0.0025

eval_dir = sim_dir / f"eval_r{cfg.rebalance_frequency}ow{cfg.optimize_window}"
eval_dir.mkdir(parents=True, exist_ok=True)

portfolio = Portfolio(annual_risk_free_rate=cfg.annual_risk_free_rate)
print(f"daily rf : {portfolio.daily_risk_free_rate:.6f}")
print(f"\neval_dir → {eval_dir}")

## 5. Run Experiment

In [ ]:
portfolios = portfolio.experiment(
    gt_returns         = gt_returns,
    gt_prices          = gt_prices,
    strategies         = {
        "genai" : mc_genai,
        "gbm"   : mc_gbm,
    },
    benchmark_strategy = cfg.benchmark_strategy,
    rebalance_freq     = cfg.rebalance_frequency,
    optimize_window    = cfg.optimize_window,
    start_date         = start_date,
    end_date           = end_date,
    transaction_cost   = cfg.transaction_cost_rate,
    init_cash          = cfg.init_cash,
    is_log_return      = cfg.is_log_return,
    method             = cfg.mu_sigma_method,
    output_dir         = str(eval_dir),
    output_suffix      = f"_{SIM_NAME}",
)

## 6. Quick Stats

In [ ]:
for name, pf in portfolios.items():
    print(f"\n── {name} ──")
    print(pf.stats())

# cumulative return plot side-by-side
portfolios["genai"].plot()
portfolios["gbm"].plot()

In [ ]:
eval_meta = {
    "sim_name"          : SIM_NAME,
    "annual_rf"         : cfg.annual_risk_free_rate,
    "rebalance_freq"    : cfg.rebalance_frequency,
    "optimize_window"   : cfg.optimize_window,
    "transaction_cost"  : cfg.transaction_cost_rate,
    "method"            : cfg.mu_sigma_method,
    "benchmark"         : cfg.benchmark_strategy,
    "start_date"        : str(start_date),
    "end_date"          : str(end_date),
    "n_assets"          : A,
    "symbols"           : symbols,
}

with open(eval_dir / "eval_meta.json", "w") as f:
    json.dump(eval_meta, f, indent=4)

# copy sim config.json เข้ามาด้วยเพื่อ traceability
shutil.copy(sim_dir / "config.json", eval_dir / "config.json")

print(f"✅ eval_meta.json saved → {eval_dir}")
print(json.dumps(eval_meta, indent=2))